# RAG 검색 전략 교체 비교 실습 (답안)

**교과목**: 고급 검색 전략과 품질 최적화  
**과제**: AI Agent 고급 검색 전략 구현 및 비교  
**권장 파일명**: `RAG_검색전략_교체비교_실습.ipynb`

동일한 Chroma 벡터스토어·동일한 TOP_N·동일한 grounded 답변 체인 위에서
**검색 전략(Retriever)만 교체**하며 결과를 비교한다.

| 전략 | 역할 | 참고 예제 |
|---|---|---|
| Basic Retrieval | 질문을 그대로 임베딩하여 similarity search | `01_HyDE를 사용한 RAG.ipynb` Naive RAG |
| HyDE | 가상 답변 문서를 생성한 뒤 그 문서로 검색 (근거는 실제 문서만) | `01_HyDE를 사용한 RAG.ipynb` |
| Multi-Query | 질문을 여러 관점으로 재작성 후 unique union | `02_Multi-Query와 Self-Query Retriever.ipynb` |
| Self-Query | 자연어에서 검색어 + `year` 메타데이터 필터를 분리 | `02_Multi-Query와 Self-Query Retriever.ipynb` |
| Reranking | BASE_K 후보를 Cross-Encoder(또는 Cohere)로 재점수화하여 TOP_N | `03_Reranking (Cross-Encoder와 Cohere Rerank).ipynb` |
| 히스토리 압축 | 오래된 턴 요약 + 최근 턴 원문 유지 | `04_대화 히스토리 압축 및 컨텍스트 관리.ipynb` |

**대상 문서** (`pdf_data/`, 농촌진흥청 농촌진흥사업 연차보고서 3종)

- `2022년도_농촌진흥사업_연차보고서.pdf`
- `2023년도_농촌진흥사업_연차보고서_내지(최종).pdf`
- `2024년도_농촌진흥사업_연차보고서.pdf`

구현 스타일은 상위 단원 노트북과, 농업 RAG 답안예제(한글 경로 시 persist 폴더 ASCII 분리, grounded 프롬프트)를 함께 참고했다.


## 0. 환경 설정

`C:\env\.env`에서 `OPENAI_API_KEY`, (선택) `COHERE_API_KEY`를 읽는다.  
키 값은 출력하지 않는다.


In [1]:
import os
import time
import hashlib
import re
import tempfile
import warnings
from pathlib import Path

from dotenv import load_dotenv

warnings.filterwarnings("ignore")

ENV_PATH = Path(r"C:\env\.env")
loaded = load_dotenv(dotenv_path=ENV_PATH, override=True)
print(f"[env] loaded={loaded} path={ENV_PATH}")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY 가 C:\\env\\.env 에 없습니다."
print("OPENAI_API_KEY: 로드 완료 (값은 출력하지 않음)")
print("COHERE_API_KEY:", "로드 완료" if COHERE_API_KEY else "(없음 - Cross-Encoder Rerank만 사용)")


[env] loaded=True path=C:\env\.env
OPENAI_API_KEY: 로드 완료 (값은 출력하지 않음)
COHERE_API_KEY: 로드 완료


In [2]:
# ---- 모델·검색 파라미터 (모든 전략 비교에서 동일하게 고정) ----
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

def resolve_notebook_dir() -> Path:
    """cwd 또는 상위 폴더에서 pdf_data/ 를 찾는다."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pdf_data").is_dir() and list((p / "pdf_data").glob("*.pdf")):
            return p
    return Path.cwd()


NOTEBOOK_DIR = resolve_notebook_dir()
PDF_DIR = NOTEBOOK_DIR / "pdf_data"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_N = 4          # 모든 전략이 최종 답변에 사용하는 문서 수
BASE_K = 12        # Rerank 1차 후보 수 (넓게 확보)

# 이미 인덱스가 있으면 재사용. True로 두면 처음부터 다시 임베딩한다.
FORCE_REINDEX = False

COLLECTION_NAME = "rda_annual_reports"


def resolve_persist_dir() -> Path:
    """Chroma/SQLite는 Windows에서 한글·특수문자 경로를 열지 못하는 경우가 있다.
    프로젝트 경로가 ASCII가 아니면 TEMP 아래 해시 폴더를 사용한다.
    (농업 문서 RAG 답안예제와 동일한 우회)
    """
    local_dir = NOTEBOOK_DIR / "chroma_db"
    if str(local_dir).isascii():
        return local_dir
    digest = hashlib.md5(str(NOTEBOOK_DIR).encode("utf-8")).hexdigest()[:10]
    return Path(tempfile.gettempdir()) / f"rda_rag_chroma_{digest}"


PERSIST_DIR = resolve_persist_dir()
print("PDF_DIR     :", PDF_DIR)
print("PERSIST_DIR :", PERSIST_DIR)
print("설정 완료")


PDF_DIR     : c:\Users\storm\Desktop\[전남ICT]생성형 AI 기반 스마트 농업 통합 서비스 개발\실습소스\05_LangChain 기반 AI Agent 활용\03_고급 검색 전략과 품질 최적화\실습과제\01_RAG 검색 전략 교체 비교 실습\pdf_data
PERSIST_DIR : C:\Users\Public\Documents\ESTsoft\CreatorTemp\rda_rag_chroma_b10d24bdd3
설정 완료


## 1. 데이터 준비: 로딩 → 청킹 → 메타데이터

- 파일명에서 연도(`year`)를 정규식으로 추출한다. 예: `2024년도_농촌진흥사업_연차보고서.pdf` → `2024`
- `source`(파일명), `page`(0-index), `year`를 메타데이터로 저장한다.
- 빈 페이지·너무 짧은 청크는 제외한다.
- 2023 PDF는 서브셋 폰트/CID 때문에 추출 품질이 떨어질 수 있다. 가능한 경우 PyMuPDF를 우선 사용한다.


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def extract_year(filename: str) -> int:
    m = re.search(r"(20\d{2})", filename)
    if not m:
        raise ValueError(f"파일명에서 연도를 찾을 수 없습니다: {filename}")
    return int(m.group(1))


def _page_index(meta: dict) -> int:
    for key in ("page", "page_number"):
        if key in meta and meta[key] is not None:
            try:
                return int(meta[key])
            except (TypeError, ValueError):
                continue
    return -1


def _load_pdf_pages(path: str) -> list[Document]:
    """PyMuPDF → PyPDF 순으로 페이지 Document를 로드한다."""
    try:
        from langchain_community.document_loaders import PyMuPDFLoader
        return PyMuPDFLoader(path).load()
    except Exception as e:
        print(f"  [warn] PyMuPDF 로드 실패, PyPDF로 재시도: {e}")
        from langchain_community.document_loaders import PyPDFLoader
        return PyPDFLoader(path).load()


def load_and_split_pdfs(pdf_dir: Path, chunk_size: int, chunk_overlap: int) -> list[Document]:
    pdf_paths = sorted(pdf_dir.glob("*.pdf"))
    print(f"발견된 PDF 파일: {len(pdf_paths)}개")
    for p in pdf_paths:
        print(" -", p.name, f"({p.stat().st_size / 1e6:.1f} MB)")

    if len(pdf_paths) < 3:
        raise FileNotFoundError(f"pdf_data/ 에 연차보고서 PDF 3개가 필요합니다: {pdf_dir}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    all_chunks: list[Document] = []
    for path in pdf_paths:
        filename = path.name
        year = extract_year(filename)
        pages = _load_pdf_pages(str(path))

        # 공백·목차 잔여물만 있는 페이지는 제외
        usable_pages = []
        for page in pages:
            text = (page.page_content or "").strip()
            if len(text) < 40:
                continue
            page.metadata["source"] = filename
            page.metadata["year"] = year
            page.metadata["page"] = _page_index(page.metadata)
            usable_pages.append(page)

        chunks = splitter.split_documents(usable_pages)
        kept = []
        for c in chunks:
            body = (c.page_content or "").strip()
            if len(body) < 80:
                continue
            c.metadata["source"] = filename
            c.metadata["year"] = year
            c.metadata["page"] = _page_index(c.metadata)
            kept.append(c)

        all_chunks.extend(kept)
        print(f"  {filename} (year={year}): pages={len(pages)} usable={len(usable_pages)} chunks={len(kept)}")

    return all_chunks


t0 = time.time()
all_chunks = load_and_split_pdfs(PDF_DIR, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"\n총 청크 수: {len(all_chunks)}개, 소요 시간: {time.time() - t0:.1f}s")

from collections import Counter
print("연도별 청크 수:", dict(Counter(c.metadata["year"] for c in all_chunks)))
print("샘플 메타데이터:", {k: all_chunks[0].metadata.get(k) for k in ("source", "year", "page")})
print("샘플 본문(앞 220자):\n", all_chunks[0].page_content[:220])


발견된 PDF 파일: 3개
 - 2022년도_농촌진흥사업_연차보고서.pdf (7.1 MB)
 - 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf (77.9 MB)
 - 2024년도_농촌진흥사업_연차보고서.pdf (88.9 MB)
  2022년도_농촌진흥사업_연차보고서.pdf (year=2022): pages=406 usable=395 chunks=663
  2023년도_농촌진흥사업_연차보고서_내지(최종).pdf (year=2023): pages=386 usable=377 chunks=607
  2024년도_농촌진흥사업_연차보고서.pdf (year=2024): pages=393 usable=380 chunks=596

총 청크 수: 1866개, 소요 시간: 1.8s
연도별 청크 수: {2022: 663, 2023: 607, 2024: 596}
샘플 메타데이터: {'source': '2022년도_농촌진흥사업_연차보고서.pdf', 'year': 2022, 'page': 0}
샘플 본문(앞 220자):
 발간등록번호
11-1390000-000627-10
2022년도
농촌진흥사업 연차보고서
2022 
Annual Report
—
Rural 
Development 
Administration


## 2. Chroma Vector DB 구축 (로컬 Persistent)

임베딩은 `text-embedding-3-small`로 고정한다.  
이후 모든 검색 전략은 **이 벡터스토어만** 사용한다.


In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENAI_API_KEY)

collection_exists = PERSIST_DIR.exists() and any(PERSIST_DIR.iterdir())

if collection_exists and not FORCE_REINDEX:
    print("기존 Chroma DB를 로드합니다:", PERSIST_DIR)
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(PERSIST_DIR),
    )
else:
    if FORCE_REINDEX and PERSIST_DIR.exists():
        import shutil
        shutil.rmtree(PERSIST_DIR, ignore_errors=True)
    PERSIST_DIR.mkdir(parents=True, exist_ok=True)
    print("Chroma DB를 새로 생성합니다:", PERSIST_DIR)
    t0 = time.time()
    vectorstore = Chroma.from_documents(
        documents=all_chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(PERSIST_DIR),
    )
    print(f"인덱싱 완료: {time.time() - t0:.1f}s")

print("컬렉션 내 문서 수:", vectorstore._collection.count())


기존 Chroma DB를 로드합니다: C:\Users\Public\Documents\ESTsoft\CreatorTemp\rda_rag_chroma_b10d24bdd3
컬렉션 내 문서 수: 1866


## 3. 공통 컴포넌트: LLM · Grounded 답변 체인

모든 전략은 같은 LLM, 같은 TOP_N, 같은 답변 체인을 쓴다.  
차이는 **어떤 문서를 검색해 컨텍스트로 넘기는가**뿐이다.

답변은 검색 문맥에만 근거한다. 없으면 모른다고 답한다.


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=LLM_MODEL, temperature=0, api_key=OPENAI_API_KEY)

GROUNDED_SYSTEM_PROMPT = """당신은 농촌진흥청 농촌진흥사업 연차보고서 기반 QA 어시스턴트입니다.
아래 [컨텍스트]에 포함된 내용만 근거로 답변하세요.
컨텍스트에 답이 없으면 "제공된 문서에서 답을 찾을 수 없습니다"라고 답하세요.
추측하거나 일반 지식을 보태지 마세요.
답변 마지막에는 참고한 출처를 (파일명, 연도, 페이지) 형식으로 표시하세요.

[컨텍스트]
{context}
"""

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", GROUNDED_SYSTEM_PROMPT),
    ("human", "{question}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()


def format_docs(docs: list[Document]) -> str:
    parts = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "?")
        year = d.metadata.get("year", "?")
        page = d.metadata.get("page", "?")
        parts.append(f"[{i}] (source={src}, year={year}, page={page})\n{d.page_content}")
    return "\n\n".join(parts)


def generate_grounded_answer(question: str, docs: list[Document]) -> str:
    # 모든 전략이 공유하는 단일 답변 생성 함수. docs만 전략별로 달라진다.
    return answer_chain.invoke({"context": format_docs(docs), "question": question})


def preview_docs(docs: list[Document], title: str = "") -> None:
    if title:
        print(title)
    for d in docs:
        snippet = " ".join(d.page_content.split())[:80]
        print(f"  - {d.metadata.get('source')}  year={d.metadata.get('year')}  p.{d.metadata.get('page')}  | {snippet}...")


print("공통 답변 생성 체인 준비 완료 (LLM:", LLM_MODEL, ")")


공통 답변 생성 체인 준비 완료 (LLM: gpt-4o-mini )


## 4. 검색 전략 구현

각 전략은 `question -> List[Document]` 형태로 통일한다.  
최종 문서 수는 `TOP_N`으로 맞춘다.


### 4-1. Basic Retrieval (Naive)

질문을 그대로 임베딩하여 벡터 유사도 검색을 수행한다.  
표현이 문서와 비슷하면 가장 빠르고, 어휘가 다르면 관련 문서를 놓칠 수 있다.


In [6]:
SAMPLE_Q = "2024년 데이터 기반 스마트농업 확산과 고도화 성과는 무엇인가?"


def retrieve_basic(question: str, k: int = TOP_N) -> list[Document]:
    return vectorstore.similarity_search(question, k=k)


basic_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_N})
preview_docs(retrieve_basic(SAMPLE_Q), "[Basic] 검색 결과")


[Basic] 검색 결과
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.249  | ③ 평가 및 향후 계획 스마트 영농 지원체계 구축 확대 및 스마트농업 테스트베드 교육장의 통합관제 체계 지원, 분야별 스마트농업 신기술 보급사업...
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.246  | 있다. 지속 가능한 스마트농업 운영 및 확대를 위한 전문가 육성을 위해 데이터 수집·활용 등 영상플랫폼을 활용하여 2022년 54명을 대상으로 ...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.68  | 069 02 분야별 추진현황 2024 스마트농업 도원･시군센터 담당자 교육 결과 지속가능한 스마트농업 운영 및 확대를 위한 역량 강화 - (개요...
  - 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf  year=2023  p.38  | 39 제3장 2023년 농촌진흥사업 추진현황 < 이시영 / 국립농업과학원 스마트팜개발과 : 063-238-4019 > < 손재용 / 국립농업과학...


### 4-2. HyDE (Hypothetical Document Embeddings)

1. LLM이 질문에 대한 **가상 답변 문서**를 생성한다.  
2. 그 가상 문서를 임베딩하여 벡터 검색한다.  
3. **최종 답변 근거는 실제 검색 문서만** 사용한다.

상위 예제 `01_HyDE를 사용한 RAG.ipynb`의 수동 HyDE와 같은 흐름이다.  
가상 문서는 검색 쿼리 용도일 뿐, 답변 프롬프트에 넣지 않는다.


In [7]:
HYDE_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 농촌진흥청 연차보고서 문체로 가상 문서를 작성하는 도우미입니다. "
        "주어진 질문에 대해, 실제 연차보고서에 있을 법한 답변 문단을 3~5문장으로 작성하세요. "
        "사실 여부를 과도하게 검증하지 말고, 검색에 쓸 '가상의 서술형 답변'만 본문으로 출력하세요.",
    ),
    ("human", "질문: {question}"),
])
hyde_chain = HYDE_PROMPT | llm | StrOutputParser()


def retrieve_hyde(question: str, k: int = TOP_N) -> list[Document]:
    hypothetical_doc = hyde_chain.invoke({"question": question})
    # 가상 문서는 검색에만 사용. 최종 답변 근거로는 쓰지 않는다.
    return vectorstore.similarity_search(hypothetical_doc, k=k)


hypo = hyde_chain.invoke({"question": SAMPLE_Q})
print("=== 가상 문서(검색용) ===")
print(hypo)
print()
preview_docs(retrieve_hyde(SAMPLE_Q), "[HyDE] 검색 결과")


=== 가상 문서(검색용) ===
2024년 데이터 기반 스마트농업의 확산과 고도화는 농업 생산성 향상과 자원 효율성을 극대화하는 데 기여하였습니다. 특히, IoT 기술과 빅데이터 분석을 활용하여 농작물 생육 환경을 실시간으로 모니터링하고, 맞춤형 농업 솔루션을 제공함으로써 농가의 경영 효율성을 높였습니다. 또한, 스마트 농기계의 보급 확대와 함께 농업인 교육 프로그램을 통해 데이터 활용 능력을 강화하여, 농업의 디지털 전환을 가속화하는 성과를 이루었습니다. 이러한 노력은 지속 가능한 농업 실현을 위한 기반을 마련하는 데 중요한 역할을 하고 있습니다.

[HyDE] 검색 결과
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.37  | 2024년도 농촌진흥사업 연차보고서 038 ③ 평가 및 향후 계획 생산량 예측 로봇 기술의 현장실증 연구를 통해 농업인의 활용성과 편의성을 향상...
  - 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf  year=2023  p.38  | 39 제3장 2023년 농촌진흥사업 추진현황 < 이시영 / 국립농업과학원 스마트팜개발과 : 063-238-4019 > < 손재용 / 국립농업과학...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.367  | 2024년도 농촌진흥사업 연차보고서 368 가. 중점 추진과제 1 농업의 미래성장 산업화 1) 데이터 기반의 스마트농업 확산과 고도화 가) 시설...
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.249  | ③ 평가 및 향후 계획 스마트 영농 지원체계 구축 확대 및 스마트농업 테스트베드 교육장의 통합관제 체계 지원, 분야별 스마트농업 신기술 보급사업...


### 4-3. Multi-Query Retriever

질문을 여러 관점으로 재작성한 뒤 각 쿼리로 검색하고, **중복을 제거한 합집합(unique union)** 을 반환한다.  
재현율(recall)을 높이는 대신 LLM 호출·검색 횟수가 늘어난다.

`langchain-classic`의 `MultiQueryRetriever`를 사용한다 (`02` 예제와 동일).


In [8]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": TOP_N}),
    llm=llm,
    include_original=True,
)


def _unique_union(docs: list[Document]) -> list[Document]:
    seen = set()
    unique = []
    for d in docs:
        key = (d.metadata.get("source"), d.metadata.get("page"), d.page_content[:120])
        if key in seen:
            continue
        seen.add(key)
        unique.append(d)
    return unique


def retrieve_multi_query(question: str, k: int = TOP_N) -> list[Document]:
    docs = _unique_union(multi_query_retriever.invoke(question))
    return docs[:k]


preview_docs(retrieve_multi_query(SAMPLE_Q), "[Multi-Query] 합집합 상위 결과")


[Multi-Query] 합집합 상위 결과
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.249  | ③ 평가 및 향후 계획 스마트 영농 지원체계 구축 확대 및 스마트농업 테스트베드 교육장의 통합관제 체계 지원, 분야별 스마트농업 신기술 보급사업...
  - 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf  year=2023  p.38  | 39 제3장 2023년 농촌진흥사업 추진현황 < 이시영 / 국립농업과학원 스마트팜개발과 : 063-238-4019 > < 손재용 / 국립농업과학...
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.246  | 있다. 지속 가능한 스마트농업 운영 및 확대를 위한 전문가 육성을 위해 데이터 수집·활용 등 영상플랫폼을 활용하여 2022년 54명을 대상으로 ...
  - 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf  year=2023  p.32  | 33 제3장 2023년 농촌진흥사업 추진현황 ③ 평가 및 향후 계획 2023년 8월 국내 스마트팜 기업을 대상으로 현장평가회를 개최한 결과, 참...


### 4-4. Self-Query Retriever

자연어에서 **검색어(semantic query)** 와 **메타데이터 필터**(예: `year=2024`)를 LLM이 분리한다.  
연차보고서는 `year`가 있으므로 "2024년에는…" 같은 질문에서 다른 연도 노이즈를 줄일 수 있다.

`02` 예제처럼 `ChromaTranslator`를 직접 지정한다.


In [9]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_community.query_constructors.chroma import ChromaTranslator

metadata_field_info = [
    AttributeInfo(
        name="year",
        description="연차보고서가 다루는 연도. 2022, 2023, 2024 중 하나의 정수값.",
        type="integer",
    ),
    AttributeInfo(
        name="source",
        description="원본 PDF 파일명 (예: '2024년도_농촌진흥사업_연차보고서.pdf')",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="PDF 내 페이지 번호 (0-indexed)",
        type="integer",
    ),
]

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="농촌진흥청 농촌진흥사업 연차보고서의 본문 조각",
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(),
    search_kwargs={"k": TOP_N},
    enable_limit=True,
)


def retrieve_self_query(question: str, k: int = TOP_N) -> list[Document]:
    docs = self_query_retriever.invoke(question)
    return docs[:k]


preview_docs(retrieve_self_query(SAMPLE_Q), "[Self-Query] 검색 결과")

structured_query = self_query_retriever.query_constructor.invoke({"query": SAMPLE_Q})
print("\n생성된 구조화 쿼리:", structured_query)


[Self-Query] 검색 결과
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.371  | 2024년도 농촌진흥사업 연차보고서 372 미래 가축개량 기반을 강화하고, 기능성 유제품･가공식품, 도체 및 육질 평가기술, 신선도･ 안전지표 ...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.37  | 정책을 추진하고 있다. ’27년까지 시설원예･축산의 30%를(시설원예 10,000㏊, 축산 11,000호) 디지털 전환하기 위해 노력하고 있으며...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.68  | 069 02 분야별 추진현황 2024 스마트농업 도원･시군센터 담당자 교육 결과 지속가능한 스마트농업 운영 및 확대를 위한 역량 강화 - (개요...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.274  | 275 02 분야별 추진현황 다음은 경영･정책 부문 조사 결과이다. 조사에 참여한 농업경영체의 절반 이상이 올해 경영 상황이 전년보다 악화될 것...

생성된 구조화 쿼리: query='스마트농업 확산과 고도화 성과' filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2024) limit=None


### 4-5. Reranking (Cross-Encoder / Cohere)

1차로 `BASE_K` 후보를 넓게 가져온 뒤, (질문, 문서) 쌍을 재점수화하여 `TOP_N`만 남긴다.  
`03` 예제와 같이 **recall은 1차에서, precision은 리랭크에서** 담당한다.

- Cross-Encoder: 로컬, API 키 불필요 (`cross-encoder/ms-marco-MiniLM-L-6-v2`)
- Cohere Rerank: `COHERE_API_KEY`가 있을 때만 실행


In [10]:
from sentence_transformers import CrossEncoder

CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)


def retrieve_rerank_cross_encoder(
    question: str, base_k: int = BASE_K, top_n: int = TOP_N
) -> list[Document]:
    candidates = vectorstore.similarity_search(question, k=base_k)
    if not candidates:
        return []
    pairs = [(question, d.page_content) for d in candidates]
    scores = cross_encoder.predict(pairs)
    reranked = sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)
    return [d for d, _ in reranked[:top_n]]


preview_docs(retrieve_rerank_cross_encoder(SAMPLE_Q), "[Rerank/Cross-Encoder] 검색 결과")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[Rerank/Cross-Encoder] 검색 결과
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.367  | 2024년도 농촌진흥사업 연차보고서 368 가. 중점 추진과제 1 농업의 미래성장 산업화 1) 데이터 기반의 스마트농업 확산과 고도화 가) 시설...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.265  | 2024년도 농촌진흥사업 연차보고서 266 나) 청년 농업경영체 규모별･품목별 경영역량 향상 지원 ⑴ 농업인 수준별 데이터 수집 및 활용 지원 ...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.37  | 2024년도 농촌진흥사업 연차보고서 038 ③ 평가 및 향후 계획 생산량 예측 로봇 기술의 현장실증 연구를 통해 농업인의 활용성과 편의성을 향상...
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.249  | 토마토, 밀, 콩, 양파, 배추 등), 177여 농가로 시설뿐만 아니라 노지의 데이터 수집과 환경제어 연계를 추진할 계획이다. 수직형 스마트팜은...


In [11]:
retrieve_rerank_cohere = None

if COHERE_API_KEY:
    from langchain_cohere import CohereRerank

    cohere_reranker = CohereRerank(
        cohere_api_key=COHERE_API_KEY,
        model="rerank-multilingual-v3.0",
        top_n=TOP_N,
    )

    def retrieve_rerank_cohere(question: str, base_k: int = BASE_K, top_n: int = TOP_N) -> list[Document]:
        candidates = vectorstore.similarity_search(question, k=base_k)
        if not candidates:
            return []
        reranked = cohere_reranker.compress_documents(documents=candidates, query=question)
        return list(reranked[:top_n])

    preview_docs(retrieve_rerank_cohere(SAMPLE_Q), "[Rerank/Cohere] 검색 결과")
else:
    print("COHERE_API_KEY가 없어 Cohere Rerank는 생략합니다. (Cross-Encoder만 사용)")


[Rerank/Cohere] 검색 결과
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.367  | 2024년도 농촌진흥사업 연차보고서 368 가. 중점 추진과제 1 농업의 미래성장 산업화 1) 데이터 기반의 스마트농업 확산과 고도화 가) 시설...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.68  | 069 02 분야별 추진현황 2024 스마트농업 도원･시군센터 담당자 교육 결과 지속가능한 스마트농업 운영 및 확대를 위한 역량 강화 - (개요...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.37  | 2024년도 농촌진흥사업 연차보고서 038 ③ 평가 및 향후 계획 생산량 예측 로봇 기술의 현장실증 연구를 통해 농업인의 활용성과 편의성을 향상...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.265  | 2024년도 농촌진흥사업 연차보고서 266 나) 청년 농업경영체 규모별･품목별 경영역량 향상 지원 ⑴ 농업인 수준별 데이터 수집 및 활용 지원 ...


## 5. 전략 레지스트리

비교 실험을 위해 `{이름: 검색함수}`로 통일한다.


In [12]:
STRATEGIES = {
    "Basic": retrieve_basic,
    "HyDE": retrieve_hyde,
    "MultiQuery": retrieve_multi_query,
    "SelfQuery": retrieve_self_query,
    "Rerank(CrossEncoder)": retrieve_rerank_cross_encoder,
}
if retrieve_rerank_cohere is not None:
    STRATEGIES["Rerank(Cohere)"] = retrieve_rerank_cohere

print("등록된 전략:", list(STRATEGIES.keys()))


등록된 전략: ['Basic', 'HyDE', 'MultiQuery', 'SelfQuery', 'Rerank(CrossEncoder)', 'Rerank(Cohere)']


## 6. 성능 비교

동일 질문·동일 생성 프롬프트로 전략만 바꿔 비교한다.

| 질문 | 의도 |
|---|---|
| 2024년 스마트농업 성과 | 연도 명시 → Self-Query 필터 효과 |
| 가루쌀 바로미2 가공 연구 | 문서 고유 용어 → Basic/HyDE 표현 차이 |
| 청년농업인·스마트 강소농 | 여러 관점 → Multi-Query recall |


In [13]:
import pandas as pd
from IPython.display import display


def run_strategy(name: str, retrieve_fn, question: str) -> dict:
    t0 = time.time()
    docs = retrieve_fn(question)
    retrieval_time = time.time() - t0

    t0 = time.time()
    answer = generate_grounded_answer(question, docs)
    generation_time = time.time() - t0

    sources = [
        f"{d.metadata.get('source')}(y={d.metadata.get('year')}, p={d.metadata.get('page')})"
        for d in docs
    ]
    years_hit = sorted({d.metadata.get("year") for d in docs})

    return {
        "strategy": name,
        "question": question,
        "num_docs": len(docs),
        "years_hit": years_hit,
        "sources": sources,
        "retrieval_time_s": round(retrieval_time, 2),
        "generation_time_s": round(generation_time, 2),
        "total_time_s": round(retrieval_time + generation_time, 2),
        "answer": answer,
        "answer_preview": answer.replace("\n", " ")[:180],
    }


def compare_strategies(question: str) -> pd.DataFrame:
    rows = []
    for name, fn in STRATEGIES.items():
        print(f"  [실행중] {name} ...")
        rows.append(run_strategy(name, fn, question))
    return pd.DataFrame(rows)


TEST_QUESTIONS = [
    "2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?",
    "가루쌀 품종 바로미2의 가공 이용 연구는 어떻게 이루어졌나?",
    "청년농업인 육성과 스마트 강소농 지원은 어떻게 추진되었나?",
]

all_results = []
for q in TEST_QUESTIONS:
    print("\n" + "=" * 80)
    print("질문:", q)
    df = compare_strategies(q)
    all_results.append(df)
    display(df[["strategy", "num_docs", "years_hit", "retrieval_time_s", "generation_time_s", "total_time_s"]])

comparison_df = pd.concat(all_results, ignore_index=True)



질문: 2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?
  [실행중] Basic ...
  [실행중] HyDE ...
  [실행중] MultiQuery ...
  [실행중] SelfQuery ...
  [실행중] Rerank(CrossEncoder) ...
  [실행중] Rerank(Cohere) ...


,strategy,num_docs,years_hit,retrieval_time_s,generation_time_s,total_time_s
0,Basic,4,"[2022, 2023]",0.12,0.49,0.61
1,HyDE,4,"[2022, 2023, 2024]",2.14,1.28,3.42
2,MultiQuery,4,"[2022, 2023]",1.48,0.51,2.00
3,SelfQuery,4,[2024],1.00,2.95,3.95
4,Rerank(CrossEncoder),4,"[2022, 2024]",0.72,4.23,4.95
5,Rerank(Cohere),4,[2024],0.43,3.76,4.19



질문: 가루쌀 품종 바로미2의 가공 이용 연구는 어떻게 이루어졌나?
  [실행중] Basic ...
  [실행중] HyDE ...
  [실행중] MultiQuery ...
  [실행중] SelfQuery ...
  [실행중] Rerank(CrossEncoder) ...
  [실행중] Rerank(Cohere) ...


,strategy,num_docs,years_hit,retrieval_time_s,generation_time_s,total_time_s
0,Basic,4,"[2022, 2023, 2024]",0.12,2.73,2.86
1,HyDE,4,"[2022, 2023, 2024]",2.38,2.32,4.69
2,MultiQuery,4,"[2022, 2023, 2024]",1.84,2.64,4.47
3,SelfQuery,4,"[2022, 2023, 2024]",1.18,3.34,4.51
4,Rerank(CrossEncoder),4,"[2022, 2024]",0.89,2.74,3.63
5,Rerank(Cohere),4,"[2022, 2024]",0.42,2.47,2.89



질문: 청년농업인 육성과 스마트 강소농 지원은 어떻게 추진되었나?
  [실행중] Basic ...
  [실행중] HyDE ...
  [실행중] MultiQuery ...
  [실행중] SelfQuery ...
  [실행중] Rerank(CrossEncoder) ...
  [실행중] Rerank(Cohere) ...


,strategy,num_docs,years_hit,retrieval_time_s,generation_time_s,total_time_s
0,Basic,4,"[2023, 2024]",0.13,4.08,4.21
1,HyDE,4,"[2023, 2024]",2.24,3.18,5.42
2,MultiQuery,4,"[2023, 2024]",1.63,3.16,4.78
3,SelfQuery,4,"[2023, 2024]",1.07,4.53,5.60
4,Rerank(CrossEncoder),4,"[2023, 2024]",0.76,3.87,4.63
5,Rerank(Cohere),4,"[2023, 2024]",0.43,3.50,3.93


In [14]:
pd.set_option("display.max_colwidth", 220)
print("=== 질문 1 상세 (출처 / 답변 미리보기) ===")
display(
    comparison_df[comparison_df["question"] == TEST_QUESTIONS[0]][
        ["strategy", "years_hit", "sources", "answer_preview"]
    ]
)


=== 질문 1 상세 (출처 / 답변 미리보기) ===


,strategy,years_hit,sources,answer_preview
0,Basic,"[2022, 2023]","[2022년도_농촌진흥사업_연차보고서.pdf(y=2022, p=249), 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf(y=2023, p=38), 2022년도_농촌진흥사업_연차보고서.pdf(y=2022, p=246), 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf(y=2023, p=32)]",제공된 문서에서 답을 찾을 수 없습니다.
1,HyDE,"[2022, 2023, 2024]","[2023년도_농촌진흥사업_연차보고서_내지(최종).pdf(y=2023, p=38), 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf(y=2023, p=32), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=37), 2022년도_농촌진흥사업_연차보고서.pdf(y=2022, p=249)]","제공된 문서에서 2024년 데이터 기반 스마트농업 확산과 고도화에 대한 구체적인 성과는 언급되어 있지 않습니다. 따라서, 해당 내용에 대한 답변을 제공할 수 없습니다. 제공된 문서에서 답을 찾을 수 없습니다."
2,MultiQuery,"[2022, 2023]","[2022년도_농촌진흥사업_연차보고서.pdf(y=2022, p=249), 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf(y=2023, p=38), 2022년도_농촌진흥사업_연차보고서.pdf(y=2022, p=246), 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf(y=2023, p=32)]",제공된 문서에서 답을 찾을 수 없습니다.
3,SelfQuery,[2024],"[2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=371), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=37), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=68), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=274)]","2024년 데이터 기반 스마트농업 확산과 고도화에서의 성과로는 다음과 같은 내용이 있습니다: 1. 스마트농업 이론 및 실습 교육이 8~10월에 2회 진행되어 46명이 교육을 받았으며, 교육 후 기자재 관리 평가 결과가 63.0에서 80.7로, 데이터 수집 및 활용 평가 결과가 59.0에서 84.8로 각각 상승했습니다."
4,Rerank(CrossEncoder),"[2022, 2024]","[2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=367), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=37), 2022년도_농촌진흥사업_연차보고서.pdf(y=2022, p=249), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=35)]",2024년 데이터 기반 스마트농업 확산과 고도화에서의 성과는 다음과 같습니다: 1. **시설 스마트팜 농작업의 인력 수급 불안정 해소**: 생산량 예측(모니터링) 로봇의 고도화를 통해 온실 환경을 인식하고 자율주행하며 과실 상태를 매일 확인할 수 있는 시스템이 개발되었습니다. 이 시스템은 한국형 스마트팜 환경에 최적화된
5,Rerank(Cohere),[2024],"[2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=367), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=68), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=37), 2024년도_농촌진흥사업_연차보고서.pdf(y=2024, p=35)]","2024년 데이터 기반 스마트농업 확산과 고도화에서의 성과는 다음과 같습니다: 1. **시설 스마트팜 적정 생산 및 운영 관리 기술 개발**: 차세대 온실 종합관리 플랫폼을 고도화하고, 복합환경 제어 및 생육 예측 기술을 현장 실증을 통해 본격 적용하였습니다. 이를 통해 수경재배 배액 살균장치의 국산화와 특용작물 중심의"


In [15]:
print("=== 전략별 평균 소요시간 ===")
summary = (
    comparison_df.groupby("strategy")[["retrieval_time_s", "generation_time_s", "total_time_s"]]
    .mean()
    .round(2)
)
display(summary)

print("\n=== 전체 비교표 (답변 요약) ===")
display(comparison_df[["question", "strategy", "years_hit", "total_time_s", "answer_preview"]])


=== 전략별 평균 소요시간 ===


,retrieval_time_s,generation_time_s,total_time_s
strategy,,,
Basic,0.12,2.43,2.56
HyDE,2.25,2.26,4.51
MultiQuery,1.65,2.10,3.75
Rerank(Cohere),0.43,3.24,3.67
Rerank(CrossEncoder),0.79,3.61,4.40
SelfQuery,1.08,3.61,4.69



=== 전체 비교표 (답변 요약) ===


,question,strategy,years_hit,total_time_s,answer_preview
0,2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?,Basic,"[2022, 2023]",0.61,제공된 문서에서 답을 찾을 수 없습니다.
1,2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?,HyDE,"[2022, 2023, 2024]",3.42,"제공된 문서에서 2024년 데이터 기반 스마트농업 확산과 고도화에 대한 구체적인 성과는 언급되어 있지 않습니다. 따라서, 해당 내용에 대한 답변을 제공할 수 없습니다. 제공된 문서에서 답을 찾을 수 없습니다."
2,2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?,MultiQuery,"[2022, 2023]",2.00,제공된 문서에서 답을 찾을 수 없습니다.
3,2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?,SelfQuery,[2024],3.95,"2024년 데이터 기반 스마트농업 확산과 고도화에서의 성과로는 다음과 같은 내용이 있습니다: 1. 스마트농업 이론 및 실습 교육이 8~10월에 2회 진행되어 46명이 교육을 받았으며, 교육 후 기자재 관리 평가 결과가 63.0에서 80.7로, 데이터 수집 및 활용 평가 결과가 59.0에서 84.8로 각각 상승했습니다."
4,2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?,Rerank(CrossEncoder),"[2022, 2024]",4.95,2024년 데이터 기반 스마트농업 확산과 고도화에서의 성과는 다음과 같습니다: 1. **시설 스마트팜 농작업의 인력 수급 불안정 해소**: 생산량 예측(모니터링) 로봇의 고도화를 통해 온실 환경을 인식하고 자율주행하며 과실 상태를 매일 확인할 수 있는 시스템이 개발되었습니다. 이 시스템은 한국형 스마트팜 환경에 최적화된
5,2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?,Rerank(Cohere),[2024],4.19,"2024년 데이터 기반 스마트농업 확산과 고도화에서의 성과는 다음과 같습니다: 1. **시설 스마트팜 적정 생산 및 운영 관리 기술 개발**: 차세대 온실 종합관리 플랫폼을 고도화하고, 복합환경 제어 및 생육 예측 기술을 현장 실증을 통해 본격 적용하였습니다. 이를 통해 수경재배 배액 살균장치의 국산화와 특용작물 중심의"
6,가루쌀 품종 바로미2의 가공 이용 연구는 어떻게 이루어졌나?,Basic,"[2022, 2023, 2024]",2.86,"가루쌀 품종 바로미2의 가공 이용 연구는 가루쌀 첨가 밀가루(중력분) 혼합 수제비와 칼국수의 가공적성을 평가하는 방식으로 이루어졌습니다. 연구 결과, 수제비 반죽의 두께는 쌀가루 함량이 증가할수록 감소하였고, 경도 및 씹힘성이 밀가루 수제비 대비 낮은 결과를 보여 섭취 시 부드러울 것이라 기대되었습니다. 또한, 바로미2를"
7,가루쌀 품종 바로미2의 가공 이용 연구는 어떻게 이루어졌나?,HyDE,"[2022, 2023, 2024]",4.69,"가루쌀 품종 바로미2의 가공 이용 연구는 가루쌀 첨가 밀가루(중력분) 혼합 수제비와 칼국수의 가공적성을 평가하는 방식으로 이루어졌습니다. 연구 결과, 수제비 반죽의 두께는 쌀가루 함량이 증가할수록 감소하였고, 경도 및 씹힘성이 밀가루 수제비 대비 낮은 결과를 보여 섭취 시 부드러울 것이라 기대되었습니다. 또한, 바로미2를"
8,가루쌀 품종 바로미2의 가공 이용 연구는 어떻게 이루어졌나?,MultiQuery,"[2022, 2023, 2024]",4.47,"가루쌀 품종 바로미2의 가공 이용 연구는 가루쌀을 첨가한 밀가루(중력분) 혼합 수제비와 칼국수의 가공적성을 평가하는 방식으로 이루어졌습니다. 연구 결과, 수제비 반죽의 두께는 쌀가루 함량이 증가할수록 감소하였고, 경도 및 씹힘성이 밀가루 수제비 대비 낮은 결과를 보여 섭취 시 부드러울 것이라 기대되었습니다. 또한, 바로미"
9,가루쌀 품종 바로미2의 가공 이용 연구는 어떻게 이루어졌나?,SelfQuery,"[2022, 2023, 2024]",4.51,"가루쌀 품종 바로미2의 가공 이용 연구는 가루쌀 첨가 밀가루(중력분) 혼합 수제비와 칼국수의 가공적성을 평가하는 방식으로 이루어졌습니다. 연구 결과, 수제비 반죽의 두께는 쌀가루 함량이 증가할수록 감소하였고, 경도 및 씹힘성이 밀가루 수제비 대비 낮은 결과를 보여 섭취 시 부드러울 것이라 기대되었습니다. 또한, 바로미2를"


### 6-1. Self-Query 연도 필터 적중 여부

"2024년"을 명시한 질문에서 Basic은 다른 연도 청크가 섞일 수 있고,
Self-Query는 `year=2024` 필터를 걸어 정밀도를 높이는지 확인한다.


In [16]:
year_question = TEST_QUESTIONS[0]

basic_docs = retrieve_basic(year_question)
selfq_docs = retrieve_self_query(year_question)

basic_years = [d.metadata.get("year") for d in basic_docs]
selfq_years = [d.metadata.get("year") for d in selfq_docs]

print("질문:", year_question)
print("Basic 연도 분포     :", basic_years)
print("Self-Query 연도 분포:", selfq_years)
print("Self-Query가 2024만 선택:", bool(selfq_years) and all(y == 2024 for y in selfq_years))

print("\n[Basic 출처]")
preview_docs(basic_docs)
print("\n[Self-Query 출처]")
preview_docs(selfq_docs)


질문: 2024년 데이터 기반 스마트농업 확산과 고도화에서 어떤 성과가 있었나?
Basic 연도 분포     : [2022, 2023, 2022, 2023]
Self-Query 연도 분포: [2024, 2024, 2024, 2024]
Self-Query가 2024만 선택: True

[Basic 출처]
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.249  | ③ 평가 및 향후 계획 스마트 영농 지원체계 구축 확대 및 스마트농업 테스트베드 교육장의 통합관제 체계 지원, 분야별 스마트농업 신기술 보급사업...
  - 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf  year=2023  p.38  | 39 제3장 2023년 농촌진흥사업 추진현황 < 이시영 / 국립농업과학원 스마트팜개발과 : 063-238-4019 > < 손재용 / 국립농업과학...
  - 2022년도_농촌진흥사업_연차보고서.pdf  year=2022  p.246  | 있다. 지속 가능한 스마트농업 운영 및 확대를 위한 전문가 육성을 위해 데이터 수집·활용 등 영상플랫폼을 활용하여 2022년 54명을 대상으로 ...
  - 2023년도_농촌진흥사업_연차보고서_내지(최종).pdf  year=2023  p.32  | 33 제3장 2023년 농촌진흥사업 추진현황 ③ 평가 및 향후 계획 2023년 8월 국내 스마트팜 기업을 대상으로 현장평가회를 개최한 결과, 참...

[Self-Query 출처]
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.371  | 2024년도 농촌진흥사업 연차보고서 372 미래 가축개량 기반을 강화하고, 기능성 유제품･가공식품, 도체 및 육질 평가기술, 신선도･ 안전지표 ...
  - 2024년도_농촌진흥사업_연차보고서.pdf  year=2024  p.37  | 정책을 추진하고 있다. ’27년까지 시설원예･축산의 30%를(시설원예 10,000㏊, 축산 11,000호) 디지털 전환하기 위해 노력하고 있으며...
  - 

## 7. 멀티턴 대화 + 히스토리 압축

`04_대화 히스토리 압축 및 컨텍스트 관리.ipynb`의 하이브리드 방식:

- 최근 N턴은 **원문 유지**
- 그보다 오래된 턴은 **누적 요약**
- 검색 문서 컨텍스트와 대화 히스토리 토큰을 분리해 센다

압축 미적용(전체 원문) vs 압축 적용의 프롬프트 토큰·답변을 비교한다.  
검색 전략은 Basic으로 고정한다 (비교의 공정성).


In [17]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o-mini")


def count_tokens(text: str) -> int:
    return len(encoding.encode(text))


class ConversationManager:
    """오래된 턴은 요약, 최근 KEEP_RECENT_TURNS 턴은 원문을 유지하는 대화 관리자."""

    def __init__(self, retrieve_fn, keep_recent_turns: int = 3, summarize_threshold: int = 6):
        self.retrieve_fn = retrieve_fn
        self.keep_recent_turns = keep_recent_turns
        self.summarize_threshold = summarize_threshold
        self.turns: list[dict] = []
        self.summary: str = ""
        self.compression_events: list[dict] = []
        self._summarized_upto = 0

        self.summarize_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                "다음은 기존 대화 요약과 새로 추가할 대화 턴입니다. "
                "핵심 질문과 결론 위주로 3~6문장 이내의 새로운 누적 요약을 작성하세요.",
            ),
            ("human", "[기존 요약]\n{prev_summary}\n\n[추가할 턴]\n{new_turns}"),
        ])
        self.summarize_chain = self.summarize_prompt | llm | StrOutputParser()

    def _build_history_context(self) -> str:
        parts = []
        if self.summary:
            parts.append(f"[이전 대화 요약]\n{self.summary}")
        recent = self.turns[-self.keep_recent_turns:]
        for t in recent:
            parts.append(f"Q: {t['question']}\nA: {t['answer']}")
        return "\n\n".join(parts)

    def _maybe_compress(self):
        if len(self.turns) <= self.summarize_threshold:
            return
        to_compress = self.turns[: -self.keep_recent_turns]
        new_chunk = to_compress[self._summarized_upto:]
        if not new_chunk:
            return
        new_turns_text = "\n\n".join(f"Q: {t['question']}\nA: {t['answer']}" for t in new_chunk)
        before_tokens = count_tokens(self.summary)
        self.summary = self.summarize_chain.invoke({
            "prev_summary": self.summary or "(없음)",
            "new_turns": new_turns_text,
        })
        self._summarized_upto = len(to_compress)
        self.compression_events.append({
            "compressed_turns": len(new_chunk),
            "summary_tokens_before": before_tokens,
            "summary_tokens_after": count_tokens(self.summary),
        })

    def ask(self, question: str) -> dict:
        history_context = self._build_history_context()
        docs = self.retrieve_fn(question)
        doc_context = format_docs(docs)

        full_prompt = ChatPromptTemplate.from_messages([
            ("system", GROUNDED_SYSTEM_PROMPT + "\n\n[이전 대화 히스토리]\n{history}"),
            ("human", "{question}"),
        ])
        chain = full_prompt | llm | StrOutputParser()

        prompt_text = GROUNDED_SYSTEM_PROMPT.format(context=doc_context) + history_context + question
        history_tokens = count_tokens(history_context)
        doc_tokens = count_tokens(doc_context)
        prompt_tokens = count_tokens(prompt_text)

        answer = chain.invoke({"context": doc_context, "history": history_context, "question": question})
        self.turns.append({"question": question, "answer": answer})
        self._maybe_compress()

        return {
            "answer": answer,
            "prompt_tokens": prompt_tokens,
            "history_tokens": history_tokens,
            "doc_tokens": doc_tokens,
            "history_context": history_context,
        }


print("ConversationManager 정의 완료")


ConversationManager 정의 완료


In [18]:
MULTI_TURN_QUESTIONS = [
    "농촌진흥사업 기본계획은 무엇이며 왜 수립하는가?",
    "스마트농업 기술은 어떻게 확산·고도화되었나?",
    "2024년 아라온실 플랫폼의 성과는 무엇인가?",
    "탄소중립·환경친화적 농업기술에는 어떤 것이 있나?",
    "청년농업인 육성과 스마트 강소농 지원 내용은?",
    "앞서 설명한 스마트농업과 청년농업인 육성은 어떻게 연결되는가?",
    "KOPIA 등 국제협력 사업은 어떻게 추진되었나?",
]

# (A) 압축 없음: 최근 턴을 크게 잡아 사실상 전체 원문 유지
no_compress_mgr = ConversationManager(retrieve_basic, keep_recent_turns=100, summarize_threshold=10_000)

# (B) 압축: 최근 2턴 원문, 3턴 초과 시 오래된 턴 요약
compress_mgr = ConversationManager(retrieve_basic, keep_recent_turns=2, summarize_threshold=3)

no_compress_log, compress_log = [], []

for q in MULTI_TURN_QUESTIONS:
    print("멀티턴 질문:", q)
    r1 = no_compress_mgr.ask(q)
    r2 = compress_mgr.ask(q)
    no_compress_log.append({
        "question": q,
        "prompt_tokens": r1["prompt_tokens"],
        "history_tokens": r1["history_tokens"],
        "answer": r1["answer"],
    })
    compress_log.append({
        "question": q,
        "prompt_tokens": r2["prompt_tokens"],
        "history_tokens": r2["history_tokens"],
        "answer": r2["answer"],
    })

print("\n압축 이벤트:", compress_mgr.compression_events)


멀티턴 질문: 농촌진흥사업 기본계획은 무엇이며 왜 수립하는가?
멀티턴 질문: 스마트농업 기술은 어떻게 확산·고도화되었나?
멀티턴 질문: 2024년 아라온실 플랫폼의 성과는 무엇인가?
멀티턴 질문: 탄소중립·환경친화적 농업기술에는 어떤 것이 있나?
멀티턴 질문: 청년농업인 육성과 스마트 강소농 지원 내용은?
멀티턴 질문: 앞서 설명한 스마트농업과 청년농업인 육성은 어떻게 연결되는가?
멀티턴 질문: KOPIA 등 국제협력 사업은 어떻게 추진되었나?

압축 이벤트: [{'compressed_turns': 2, 'summary_tokens_before': 0, 'summary_tokens_after': 147}, {'compressed_turns': 1, 'summary_tokens_before': 147, 'summary_tokens_after': 164}, {'compressed_turns': 1, 'summary_tokens_before': 164, 'summary_tokens_after': 166}, {'compressed_turns': 1, 'summary_tokens_before': 166, 'summary_tokens_after': 163}]


In [19]:
multi_turn_df = pd.DataFrame({
    "question": [x["question"] for x in no_compress_log],
    "prompt_tokens_no_compress": [x["prompt_tokens"] for x in no_compress_log],
    "prompt_tokens_compressed": [x["prompt_tokens"] for x in compress_log],
    "history_tokens_no_compress": [x["history_tokens"] for x in no_compress_log],
    "history_tokens_compressed": [x["history_tokens"] for x in compress_log],
})
multi_turn_df["tokens_saved"] = (
    multi_turn_df["prompt_tokens_no_compress"] - multi_turn_df["prompt_tokens_compressed"]
)
display(multi_turn_df)


,question,prompt_tokens_no_compress,prompt_tokens_compressed,history_tokens_no_compress,history_tokens_compressed,tokens_saved
0,농촌진흥사업 기본계획은 무엇이며 왜 수립하는가?,1472,1472,0,0,0
1,스마트농업 기술은 어떻게 확산·고도화되었나?,1622,1648,238,264,-26
2,2024년 아라온실 플랫폼의 성과는 무엇인가?,2297,2323,630,656,-26
3,탄소중립·환경친화적 농업기술에는 어떤 것이 있나?,2801,2530,936,665,271
4,청년농업인 육성과 스마트 강소농 지원 내용은?,2859,2343,1215,699,516
5,앞서 설명한 스마트농업과 청년농업인 육성은 어떻게 연결되는가?,2805,2015,1656,866,790
6,KOPIA 등 국제협력 사업은 어떻게 추진되었나?,4233,3121,2058,946,1112


In [20]:
print("=== 압축 미적용 마지막 답변 (이전 맥락 참조형) ===")
print(no_compress_log[-2]["answer"])
print("\n=== 압축 적용 마지막 맥락 참조 답변 ===")
print(compress_log[-2]["answer"])
print("\n=== 압축 적용 시 누적 요약 ===")
print(compress_mgr.summary or "(아직 요약 없음)")


=== 압축 미적용 마지막 답변 (이전 맥락 참조형) ===
스마트농업과 청년농업인 육성은 다음과 같이 연결됩니다:

1. **기술 역량 강화**: 청년농업인 육성 프로그램은 현장 중심의 기술역량 강화를 목표로 하고 있으며, 스마트농업 기술의 도입과 활용을 통해 청년농업인들이 농업 경영에 필요한 데이터 활용 능력을 배양할 수 있도록 지원합니다. 맞춤형 교육과 R&D 실증과제 참여를 통해 청년농업인들이 스마트농업 기술을 이해하고 적용할 수 있는 기회를 제공합니다.

2. **스마트 강소농 프로그램**: 청년농업인 육성과 함께 중소규모 농가의 스마트농업 도입을 촉진하기 위해 '스마트 강소농 프로그램'이 운영됩니다. 이 프로그램은 청년농업인들이 스마트농업 기술을 배우고 실천할 수 있는 기반을 마련하여, 농업 경영의 효율성을 높이는 데 기여합니다.

3. **지속 가능한 성장 유도**: 청년농업인들이 스마트농업 기술을 통해 농업 경영의 자립도를 높이고, 안정적인 정착을 도모할 수 있도록 지원함으로써, 농업의 지속 가능성을 높이는 데 기여합니다. 이는 청년농업인들이 농업 환경 변화에 능동적으로 대처할 수 있는 능력을 키우는 데 중요한 역할을 합니다.

이러한 방식으로 스마트농업과 청년농업인 육성은 상호 보완적으로 작용하여 농업의 미래를 이끌어갈 인재를 양성하는 데 기여하고 있습니다. (2024년도_농촌진흥사업_연차보고서.pdf, 2024, 페이지 265, 375)

=== 압축 적용 마지막 맥락 참조 답변 ===
스마트농업과 청년농업인 육성은 다음과 같이 연결됩니다:

1. **기술 역량 강화**: 청년농업인 육성을 위한 맞춤형 교육과 R&D 실증과제 참여를 통해 청년농업인의 현장 중심 기술역량을 키우고, 스마트농업 기술을 이해하고 활용할 수 있도록 지원합니다. 이를 통해 청년농업인들이 스마트농업을 도입하고 실천할 수 있는 기반을 마련합니다.

2. **스마트농업 지원 프로그램**: 청년농업인들이 스마트농업 기술을 활용하여 운영 효율성을 제고할 수 있도록 '스마트 강소

## 8. 정리: 상황별 추천 전략

동일 코퍼스·동일 생성 체인에서 **검색 전략만** 바꿨을 때의 해석이다.
실행 결과 표(6절, 6-1절, 7절)와 함께 본다.

| 상황 | 추천 전략 | 이유 |
|---|---|---|
| 질문이 짧고 문서 용어와 비슷함 | **Basic** | 추가 LLM 호출 없이 가장 빠르고 저렴 |
| 질문 표현이 보고서 문체와 다름 (추상·구어 질문) | **HyDE** | 가상 답변 문서로 질문-문서 임베딩 간극을 줄임. 가상 문서는 검색용만 |
| 한 질문이 여러 하위 주제(육성/교육/지원)를 포함 | **Multi-Query** | 재작성 쿼리 합집합으로 recall 향상. 비용·시간은 증가 |
| "2024년에는…"처럼 연도 조건이 명시됨 | **Self-Query** | `year` 필터를 자동 추출해 다른 연도 노이즈를 배제 |
| 후보를 넓게 모은 뒤 상위 정확도가 중요 | **Rerank** | BASE_K → Cross-Encoder/Cohere로 precision 향상. 지연 증가 |
| 대화가 길어지는 멀티턴 | **히스토리 압축** | 오래된 턴 요약 + 최근 턴 원문으로 토큰을 줄이면서 맥락 유지 |

**선택 가이드**

- 속도·비용이 우선이면 Basic으로 시작한다.
- 연도가 중요한 연차보고서 QA에서는 Self-Query를 기본으로 두고, 필요 시 Rerank를 뒤에 붙인다.
- 질문이 모호하거나 주제가 넓으면 Multi-Query 또는 HyDE로 1차 recall을 올린 뒤 Rerank로 정리한다.
- 2023 PDF처럼 추출 품질이 낮으면 해당 연도 적중이 떨어질 수 있다. 메타데이터 필터(Self-Query)가 텍스트 품질 이슈를 일부 보완한다.
- 실무 파이프라인 예: `Self-Query(필터) → 넓은 k → Rerank(TOP_N) → grounded 생성` + 멀티턴 요약 압축.

**공정 비교를 지킨 점**

- 임베딩(`text-embedding-3-small`)·생성(`gpt-4o-mini`)·`TOP_N=4`·답변 프롬프트를 고정했다.
- 차이는 Retriever/검색 전략과 (Rerank의) 1차 `BASE_K`뿐이다.
- 답변은 검색 문맥에만 근거하도록 grounded 프롬프트를 사용했다.
